# Fig. 5b and Sup. Fig. 11a: GSDensity base scores of the Visium HD mouse brain

Per-bin GSDensity relevance (MCA co-embedding, 300-NN graph, random walk with restart; nn_use 300, restart 0.75, MCA
dimensions 1-10, seed 0; spatial KLD n = 10, 100 permutations) of the pyramidal layer (top 20) and striatum (top 50)
signatures on all 98,917 16-um bins, run in the R environment gsdensity_env. One CSV per signature is written to
data/gsd_cache/visiumhd/ for the figure notebooks.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"
os.makedirs(source_dir, exist_ok=True)

# GSDensity scores of this dataset (one CSV per signature) and the R task directory
gsd_dir = working_dir + "data/gsd_cache/visiumhd/"
task_dir = working_dir + "data/rev_cache/GSD_base_visiumhd/task_full/"

In [5]:
from helpers import (
    export_gsdensity_task,
    run_gsdensity,
    read_gsdensity_outputs,
    write_gsdensity_cache,
)

In [6]:
# 1. Load data (02_visiumhd_platform cell 29: 16-um bins, log-normalised X)
adata = sc.read_h5ad(processed_data + "visiumhd_pyramidal_layer_scores.h5ad")

# MCA's fuzzy coding divides by the gene range, so genes with no non-zero value are dropped
detected = np.asarray((adata.X > 0).sum(axis=0)).ravel() > 0
adata = adata[:, detected].copy()
print(f"{adata.n_obs} bins x {adata.n_vars} genes")

98917 bins x 9952 genes


In [7]:
# 2. Gene sets (02 cell 9; Fig. 5b, Sup. Fig. 11a)
with open(project_dir + "Pyramidal_layer_signatures.pkl", "rb") as f:
    signature_dict_pyramidal = pickle.load(f)
with open(project_dir + "Striatum_signatures.pkl", "rb") as f:
    signature_dict_striatum = pickle.load(f)

signatures_dict = {
    "Pyramidal_layer": list(signature_dict_pyramidal["Pyramidal_layer_top20"]),
    "Striatum": list(signature_dict_striatum["Striatum_top50"]),
}
for s, genes in signatures_dict.items():
    print(f"{s}: {sum(g in adata.var_names for g in genes)}/{len(genes)} genes in the data")

Pyramidal_layer: 13/20 genes in the data
Striatum: 40/50 genes in the data


In [8]:
# 3. GSDensity bridge: R package in the conda environment gsdensity_env, run by scripts/run_gsdensity.R
gsd_rscript = working_dir + "scripts/run_gsdensity.R"
gsd_params = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0, "spatial": True, "spatial_n": 10,
              "spatial_ntimes": 100}

# GSDensity passes the whole gene-bin graph (about 711 MiB here) through future.apply, above future's 500 MiB size
# guard; the guard (a size check only) is raised for the R session through this environment variable
os.environ["R_FUTURE_GLOBALS_MAXSIZE"] = str(16 * 1024**3)

In [9]:
# 4. One task for the slide (both gene sets share one MCA embedding and graph), run in one R session
export_gsdensity_task(adata, signatures_dict, task_dir, gsd_params)
t0 = time.time()
rc = run_gsdensity([task_dir], gsd_rscript, n_jobs=1, log=working_dir + "logs/REV_GSD_base_visiumhd_R.log")
print(f"R exit code: {rc}; wall clock: {time.time() - t0:.0f} s")

R exit code: [0]; wall clock: 943 s


`dnet::dRWR` (inside GSDensity's random walk) sets steady-state probabilities below 1e-6 to zero. On a graph of
about 100,000 spots or cells plus genes, most of them therefore tie at relevance 0; this is native GSDensity behaviour
and is counted in the index (`n_zero_relevance`).

In [10]:
# 5. One CSV per signature and the index
gsd = read_gsdensity_outputs(task_dir)
index = write_gsdensity_cache(
    adata,
    gsd,
    signatures_dict,
    gsd_dir,
    working_dir + "data/gsd_cache/INDEX_visiumhd.csv",
    dataset="visiumhd",
    params=gsd_params,
    note=f"none: full data, all {adata.n_obs:,} 16-um bins",
    n_total=adata.n_obs,
    working_dir=working_dir,
)
print(index[["signature", "genes_used_of_given", "n_spots", "n_positive_label", "n_zero_relevance", "runtime_s",
             "spatial_kld_p"]].to_string(index=False))

      signature genes_used_of_given  n_spots  n_positive_label  n_zero_relevance  runtime_s  spatial_kld_p
Pyramidal_layer               13/20    98917                57             92930      915.4            NaN
       Striatum               40/50    98917                64             87215      915.4            NaN
